In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from utils import balance_by_min_class

In [2]:
# 1. Use GPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [ ]:
# Load the pre-existing deduplicated splits
train_df = pd.read_csv('../data/train_data.csv').dropna(subset=['content', 'Sentiment'])
test_df = pd.read_csv('../data/test_data.csv').dropna(subset=['content', 'Sentiment'])

# Create a validation set out of the training split
train_df, val_df = train_test_split(train_df, test_size=0.125, random_state=42, stratify=train_df['Sentiment'])

# Balance the training set by downsampling every class to the smallest.
print("Before balancing:", train_df['Sentiment'].value_counts().to_dict())
train_df = balance_by_min_class(train_df, label_col='Sentiment', random_state=42)

print(f"\nTrain: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print("Train counts (Balanced):", train_df['Sentiment'].value_counts().to_dict())

Before balancing: {'Negative': 5729, 'Neutral': 4520, 'Positive': 4116}

Train: 12348 | Val: 2053 | Test: 4105
Train counts (Balanced): {'Neutral': 4116, 'Positive': 4116, 'Negative': 4116}


In [ ]:
# Sanity checks to prevent leakage
def _make_keys(df):
    return set(zip(df['appName'].astype(str), df['content'].astype(str)))

for name, df_ in [('train', train_df), ('val', val_df), ('test', test_df)]:
    assert df_[['appName', 'content', 'clean_content', 'Sentiment']].isna().any(axis=1).sum() == 0, f"Missing core fields in {name}"
    assert (df_['content'].astype(str).str.strip() == '').sum() == 0, f"Empty content in {name}"

train_keys = _make_keys(train_df)
val_keys = _make_keys(val_df)
test_keys = _make_keys(test_df)

assert len(train_keys & val_keys) == 0, "Train/Val overlap detected"
assert len(train_keys & test_keys) == 0, "Train/Test overlap detected"
assert len(val_keys & test_keys) == 0, "Val/Test overlap detected"

print("Sanity checks passed")
print(f"Train rows: {len(train_df)} | Val rows: {len(val_df)} | Test rows: {len(test_df)}")
print("Train label counts:", train_df['Sentiment'].value_counts().to_dict())
print("Val label counts:", val_df['Sentiment'].value_counts().to_dict())
print("Test label counts:", test_df['Sentiment'].value_counts().to_dict())

Sanity checks passed
Train rows: 12348 | Val rows: 2053 | Test rows: 4105
Train label counts: {'Neutral': 4116, 'Positive': 4116, 'Negative': 4116}
Val label counts: {'Negative': 819, 'Neutral': 646, 'Positive': 588}
Test label counts: {'Negative': 1637, 'Neutral': 1292, 'Positive': 1176}


In [5]:
# Convert text labels (Positive/Negative/Neutral) to numbers (0, 1, 2)
le = LabelEncoder()
train_labels = le.fit_transform(train_df['Sentiment'])
val_labels = le.transform(val_df['Sentiment'])
test_labels = le.transform(test_df['Sentiment'])

In [6]:
# Load the DeBERTa-v3 Tokenizer (SentencePiece-based, loaded via AutoTokenizer)
MODEL_NAME = 'microsoft/deberta-v3-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [7]:
# Tokenize raw text with fixed-length padding so train/val/test tensors match shape.
train_encodings = tokenizer(train_df['content'].tolist(), truncation=True, padding='max_length', max_length=128)
val_encodings   = tokenizer(val_df['content'].tolist(),   truncation=True, padding='max_length', max_length=128)
test_encodings  = tokenizer(test_df['content'].tolist(),  truncation=True, padding='max_length', max_length=128)

In [8]:
# Create a Custom PyTorch Dataset
class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = ReviewDataset(train_encodings, train_labels)
val_dataset = ReviewDataset(val_encodings, val_labels)
test_dataset = ReviewDataset(test_encodings, test_labels)

In [ ]:
# Load the pre-trained DeBERTa-v3 model (configured for 3 labels)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
model = model.float()
model.to(device)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias          

DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): Layer

In [ ]:
# Define the evaluation metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels, preds, average='macro')
    precision = precision_score(labels, preds, average='macro')
    recall = recall_score(labels, preds, average='macro')
    return {'macro_f1': f1, 'macro_precision': precision, 'macro_recall': recall}

In [ ]:
# warmup_steps computed explicitly from the training set size
batch_size = 4
grad_acc_steps = 4
effective_batch_size = batch_size * grad_acc_steps
num_epochs = 10
steps_per_epoch = -(-len(train_dataset) // effective_batch_size) 
total_steps = steps_per_epoch * num_epochs
warmup_steps = int(0.1 * total_steps)
print(f'Training steps: {total_steps} (warmup: {warmup_steps})')

training_args = TrainingArguments(
    output_dir='./results_balanced_deberta',
    num_train_epochs=num_epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=grad_acc_steps,
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=1e-5,
    weight_decay=0.01,
    warmup_steps=warmup_steps,
    max_grad_norm=1.0,
    optim="adamw_torch",
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    save_total_limit=2,
    report_to='none',
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Starting Training...")
trainer.train()

Training steps: 7720 (warmup: 772)
Starting Training...


/Users/jakobaune/anaconda3/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Macro F1,Macro Precision,Macro Recall
1,1.006491,0.801012,0.650494,0.672230,0.669217
2,0.723661,0.763736,0.666370,0.675617,0.680236
3,0.649788,0.752475,0.673212,0.675825,0.682765


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/jakobaune/anaconda3/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/jakobaune/anaconda3/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Evaluation on the Test Set

print("\n--- FINAL DEBERTA-V3 RESULTS ---")
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

macro_f1 = f1_score(labels, preds, average='macro')
macro_precision = precision_score(labels, preds, average='macro')
macro_recall = recall_score(labels, preds, average='macro')

print(f"Macro F1-score: {macro_f1:.4f}")
print(f"Macro Precision: {macro_precision:.4f}")
print(f"Macro Recall: {macro_recall:.4f}")

print("\nDetailed Classification Report:")
print(classification_report(labels, preds, target_names=le.classes_))